In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/NguyenAn05/pocket-r1.git'
REPO_DIR = '/kaggle/working/pocket-r1'

print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], text=True))

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'])
print('Setup complete!')


In [ ]:
import subprocess, time, urllib.request, sys

MODEL = 'Qwen/Qwen3-4B'
LOG_PATH = '/kaggle/working/vllm_server.log'
log_file = open(LOG_PATH, 'w')

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--dtype', 'half',
    '--tensor-parallel-size', '2',
    '--max-model-len', '2048',
    '--gpu-memory-utilization', '0.85',
    '--enforce-eager',
    '--host', '127.0.0.1',
    '--port', '8000',
]
print('Starting vLLM:', ' '.join(vllm_cmd))
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

ready = False
for sec in range(420):
    if vllm_proc.poll() is not None:
        log_file.flush()
        with open(LOG_PATH) as f:
            print('vLLM exited early! Last logs:\n', f.read()[-4000:])
        raise RuntimeError(f'vLLM crashed with code {vllm_proc.returncode}')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2) as r:
            if r.status == 200:
                print(f'vLLM ready in {sec}s:', r.read().decode())
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    vllm_proc.terminate()
    raise TimeoutError('vLLM did not become ready within 420s')


In [ ]:
import subprocess, sys, os

OUT_DIR = '/kaggle/working/exp0_baseline'
os.makedirs(OUT_DIR, exist_ok=True)

try:
    cmd = [
        sys.executable, f'{REPO_DIR}/scripts/run_exp0_baseline.py',
        '--config', f'{REPO_DIR}/configs/eval/exp0_baseline.yaml',
        '--backend', 'openai_compatible',
        '--model', 'Qwen/Qwen3-4B',
        '--base-url', 'http://127.0.0.1:8000/v1',
        '--workers', '24',
        '--no-ev',
        '--output-dir', OUT_DIR,
    ]
    print('Running Exp 0:', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=REPO_DIR)
finally:
    print('Shutting down vLLM server...')
    vllm_proc.terminate()
    try:
        vllm_proc.wait(timeout=20)
    except Exception:
        vllm_proc.kill()
    log_file.close()
    print('Done! Output files:', os.listdir(OUT_DIR))
